In [ ]:
import pandas as pd
from scipy.spatial import cKDTree

# 1. Load data
collisions = pd.read_csv('enriched_collision_data_with_weather.csv')
mapping = pd.read_csv('metadata_images_mapping.csv')

# Drop missing coordinate rows
collisions = collisions.dropna(subset=['latitude', 'longitude'])
mapping = mapping.dropna(subset=['lat', 'lng'])

# 2. Build KDTree for quick spatial distance lookups
tree = cKDTree(mapping[['lat', 'lng']].values)

# Query nearest image for each collision
distances, indices = tree.query(collisions[['latitude', 'longitude']].values)

# 3. Clean up the image name for the URL
def extract_image_name(blob_name):
    # e.g., "training/training/Images/2834_0.jpg" -> "2834_0.jpg"
    return str(blob_name).split('/')[-1]

collisions['closest_image'] = mapping.iloc[indices]['image_blob_name'].apply(extract_image_name).values

# 4. Save the combined dataset
collisions.to_csv('processed_collisions_with_images.csv', index=False)